
# Day 3 — PyTorch Modules

## Goal
Learn:

- `torch.nn.Module`
- `nn.Linear`
- Activation functions such as `nn.ReLU`
- The `forward()` method
- Input/output shapes
- Trainable parameters
- How Day 2 Autograd connects to neural-network modules
- How to build a simple two-layer classifier

Run the notebook **top to bottom**.


In [1]:
import torch
import torch.nn as nn

print('PyTorch version:', torch.__version__)

PyTorch version: 2.11.0+cpu



## 1. What is `nn.Module`?

In PyTorch, neural networks are usually created by defining a class that inherits from:

```python
nn.Module
```

A custom model normally has:

1. `__init__()` — define the layers
2. `forward()` — define how data moves through the layers


In [2]:

class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return x

model = SimpleModel()
print(model)


SimpleModel()



## 2. Linear Layers

A linear layer can be written as:

\[
y = xW^T + b
\]

Example:

```python
nn.Linear(4, 3)
```

means:

- 4 input features
- 3 output features


In [3]:

layer = nn.Linear(4, 3)

print(layer)
print("Weight shape:", layer.weight.shape)
print("Bias shape:", layer.bias.shape)


Linear(in_features=4, out_features=3, bias=True)
Weight shape: torch.Size([3, 4])
Bias shape: torch.Size([3])


## 3. Passing Data Through a Layer

In [4]:

x = torch.tensor([[1.0, 2.0, 3.0, 4.0]])

output = layer(x)

print("Input shape:", x.shape)
print("Output:", output)
print("Output shape:", output.shape)


Input shape: torch.Size([1, 4])
Output: tensor([[ 0.6238, -0.2306,  1.3643]], grad_fn=<AddmmBackward0>)
Output shape: torch.Size([1, 3])



## 4. Activation Functions

A common activation function is:

```python
nn.ReLU()
```

ReLU keeps positive values and converts negative values to zero:

\[
ReLU(x)=\max(0,x)
\]


In [5]:

relu = nn.ReLU()

values = torch.tensor([-3.0, -1.0, 0.0, 2.0, 5.0])

print("Before ReLU:", values)
print("After ReLU :", relu(values))


Before ReLU: tensor([-3., -1.,  0.,  2.,  5.])
After ReLU : tensor([0., 0., 0., 2., 5.])



# 5. Build a Two-Layer Classifier

Architecture:

```text
Input
  ↓
Linear Layer 1
  ↓
ReLU
  ↓
Linear Layer 2
  ↓
Output logits
```

We will use:

- Input features = 4
- Hidden neurons = 8
- Output classes = 3


In [6]:

class TwoLayerClassifier(nn.Module):
    def __init__(self, input_size=4, hidden_size=8, num_classes=3):
        super().__init__()

        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

model = TwoLayerClassifier()
print(model)


TwoLayerClassifier(
  (fc1): Linear(in_features=4, out_features=8, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=8, out_features=3, bias=True)
)


## 6. Forward Pass

In [7]:

x = torch.tensor([[1.0, 2.0, 3.0, 4.0]])

output = model(x)

print("Input shape:", x.shape)
print("Output:", output)
print("Output shape:", output.shape)


Input shape: torch.Size([1, 4])
Output: tensor([[ 0.5729, -1.1055,  0.0340]], grad_fn=<AddmmBackward0>)
Output shape: torch.Size([1, 3])



The final output contains **3 logits**, one for each class.

When you call:

```python
model(x)
```

PyTorch automatically uses the model's `forward()` method.


## 7. Predicted Class and Probabilities

In [8]:

logits = model(x)

predicted_class = torch.argmax(logits, dim=1)
probabilities = torch.softmax(logits, dim=1)

print("Logits:", logits)
print("Predicted class:", predicted_class)
print("Probabilities:", probabilities)
print("Probability sum:", probabilities.sum(dim=1))


Logits: tensor([[ 0.5729, -1.1055,  0.0340]], grad_fn=<AddmmBackward0>)
Predicted class: tensor([0])
Probabilities: tensor([[0.5650, 0.1055, 0.3296]], grad_fn=<SoftmaxBackward0>)
Probability sum: tensor([1.], grad_fn=<SumBackward1>)



For training with `nn.CrossEntropyLoss()`, do **not** apply softmax manually before the loss function.


## 8. Process a Batch

In [9]:

batch = torch.randn(5, 4)

output = model(batch)

print("Input batch shape :", batch.shape)
print("Output batch shape:", output.shape)


Input batch shape : torch.Size([5, 4])
Output batch shape: torch.Size([5, 3])



Expected:

```text
Input : [5, 4]
Output: [5, 3]
```


## 9. Inspect Trainable Parameters

In [10]:

for name, parameter in model.named_parameters():
    print(name)
    print("Shape:", parameter.shape)
    print("Requires gradient:", parameter.requires_grad)
    print()


fc1.weight
Shape: torch.Size([8, 4])
Requires gradient: True

fc1.bias
Shape: torch.Size([8])
Requires gradient: True

fc2.weight
Shape: torch.Size([3, 8])
Requires gradient: True

fc2.bias
Shape: torch.Size([3])
Requires gradient: True



## 10. Count Model Parameters

In [11]:

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)


Total parameters: 67
Trainable parameters: 67



For this model:

- First layer: \(4 \times 8 + 8 = 40\)
- Second layer: \(8 \times 3 + 3 = 27\)

Total:

\[
40 + 27 = 67
\]

trainable parameters.


## 11. Build the Same Network With `nn.Sequential`

In [12]:

sequential_model = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 3)
)

print(sequential_model)

x = torch.randn(2, 4)
print("Output shape:", sequential_model(x).shape)


Sequential(
  (0): Linear(in_features=4, out_features=8, bias=True)
  (1): ReLU()
  (2): Linear(in_features=8, out_features=3, bias=True)
)
Output shape: torch.Size([2, 3])



## 12. Connect Day 2 Autograd to Day 3 Modules

Parameters inside `nn.Module` automatically have:

```python
requires_grad=True
```

Therefore Autograd can calculate gradients for all trainable weights and biases.


In [13]:

model = TwoLayerClassifier()

x = torch.randn(2, 4)
labels = torch.tensor([0, 2])

logits = model(x)

criterion = nn.CrossEntropyLoss()
loss = criterion(logits, labels)

print("Loss:", loss)

loss.backward()

for name, parameter in model.named_parameters():
    print(name, "gradient shape:", parameter.grad.shape)


Loss: tensor(1.1915, grad_fn=<NllLossBackward0>)
fc1.weight gradient shape: torch.Size([8, 4])
fc1.bias gradient shape: torch.Size([8])
fc2.weight gradient shape: torch.Size([3, 8])
fc2.bias gradient shape: torch.Size([3])



The flow is now:

```text
Input
  ↓
Model
  ↓
Logits
  ↓
Loss
  ↓
loss.backward()
  ↓
Gradients for every trainable parameter
```



# 13. Day 3 Main Exercise

Build a classifier with:

- Input features = 10
- Hidden neurons = 16
- Output classes = 4
- Activation = ReLU

Architecture:

```text
10 inputs
   ↓
Linear(10, 16)
   ↓
ReLU
   ↓
Linear(16, 4)
   ↓
4 logits
```


In [14]:

class MyClassifier(nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(10, 16)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(16, 4)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

my_model = MyClassifier()

x = torch.randn(6, 10)
output = my_model(x)

print(my_model)
print("Input shape:", x.shape)
print("Output shape:", output.shape)

assert output.shape == (6, 4)

print("\nTwo-layer classifier passed successfully!")


MyClassifier(
  (fc1): Linear(in_features=10, out_features=16, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=16, out_features=4, bias=True)
)
Input shape: torch.Size([6, 10])
Output shape: torch.Size([6, 4])

Two-layer classifier passed successfully!


## 14. Inspect Intermediate Shapes

In [15]:

x = torch.randn(6, 10)

hidden = my_model.fc1(x)
activated = my_model.relu(hidden)
output = my_model.fc2(activated)

print("Input shape   :", x.shape)
print("After fc1    :", hidden.shape)
print("After ReLU   :", activated.shape)
print("Final output :", output.shape)


Input shape   : torch.Size([6, 10])
After fc1    : torch.Size([6, 16])
After ReLU   : torch.Size([6, 16])
Final output : torch.Size([6, 4])



# Day 3 Summary

You learned how to:

- Create models with `nn.Module`
- Define layers in `__init__()`
- Define a forward pass with `forward()`
- Use `nn.Linear`
- Use `nn.ReLU`
- Work with batches
- Understand logits and probabilities
- Inspect trainable model parameters
- Count parameters
- Use `nn.Sequential`
- Connect Autograd to model parameters
- Use `loss.backward()` with a neural network
- Build a two-layer classifier

## Core Mental Model

```text
Input tensor
     ↓
Linear layer
     ↓
Activation
     ↓
Linear layer
     ↓
Logits
     ↓
Loss
     ↓
Backward pass
     ↓
Parameter gradients
```

## Next Topic

**Loss Functions + Optimizers + Full Training Loop**

That is where the classifier begins to actually learn from data.
